# 18 — Networking, HTTP, and APIs

Goal: consume and build HTTP APIs safely, and understand networking basics.

_Generated: 2026-02-19_

## Setup

This course targets **Python 3.11+** (works on 3.10+, with a few feature differences).

Recommended tooling:

```bash
# create + activate a virtual environment
python -m venv .venv
# mac/linux:
source .venv/bin/activate
# windows (PowerShell):
# .venv\Scripts\Activate.ps1

python -m pip install -U pip

# quality-of-life (optional but recommended)
python -m pip install -U ipykernel ruff black pytest mypy
python -m pip install -U requests httpx
```

If you're using Jupyter:
```bash
python -m ipykernel install --user --name python-course --display-name "Python Course (.venv)"
```

In [ ]:

import sys, platform, os
print("python:", sys.version.split()[0])
print("implementation:", platform.python_implementation())
print("platform:", platform.platform())
print("cwd:", os.getcwd())


## 1.
L1: HTTP in practice

Key ideas:
- method: GET/POST/PUT/PATCH/DELETE
- status codes: 2xx ok, 4xx client error, 5xx server error
- headers + body
- idempotency and retries (safe to retry GET; careful with POST)

## 2.
L2: Build a tiny local HTTP server (stdlib)

We’ll serve JSON from localhost, then call it as a client.

In [ ]:

import threading, json, time
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.request import urlopen

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/health":
            body = {"status": "ok"}
        else:
            body = {"path": self.path, "msg": "hello"}
        data = json.dumps(body).encode("utf-8")
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def log_message(self, fmt, *args):
        # silence server logs in notebook
        return

server = HTTPServer(("127.0.0.1", 0), Handler)  # port 0 => pick a free port
host, port = server.server_address

t = threading.Thread(target=server.serve_forever, daemon=True)
t.start()

url = f"http://{host}:{port}/health"
with urlopen(url, timeout=2) as r:
    payload = json.loads(r.read().decode("utf-8"))
print("GET", url, "->", payload)

server.shutdown()
server.server_close()


## 3.
L3: Robust clients: timeouts, retries, backoff

Rules:
- always set timeouts
- retry on transient failures (connection resets, 502/503/504)
- never retry non-idempotent requests blindly

## 4.
L4: JSON APIs and schema validation

For non-trivial APIs, validate responses:
- `pydantic` models (third-party)
- `jsonschema` (third-party)
- TypedDict/dataclasses + manual checks (stdlib)

## 5.
L5: Sockets (very low-level)

Most Python work uses HTTP libraries; sockets are for special cases.

In [ ]:

import socket

# resolve a hostname (no outbound connection needed)
info = socket.getaddrinfo("localhost", 80)
print("addrinfo entries:", len(info))
print("example:", info[0][:3])


## 6.
L6: Exercises

1. Extend the local server with a `/time` endpoint.
2. Write a client function `get_json(url) -> dict` with timeout and error handling.
3. (Optional) Rewrite the client using `requests` or `httpx`.

## 7.
L7: Retry with exponential backoff (template)

This is a *pattern*, not a copy-paste. Adjust for your API constraints.

In [ ]:

import time
from collections.abc import Callable

def retry_backoff(fn: Callable[[], object], attempts: int = 5, base: float = 0.05):
    last = None
    for i in range(attempts):
        try:
            return fn()
        except Exception as e:
            last = e
            time.sleep(base * (2 ** i))
    raise RuntimeError("all retries failed") from last

# demo with a flaky callable
i = 0
def flaky():
    global i
    i += 1
    if i < 3:
        raise OSError("temporary")
    return "ok"

print(retry_backoff(flaky))


## 8.
L8: HTTP client sessions (requests/httpx)

When you use `requests` or `httpx`, prefer a session/client:
- connection pooling
- shared headers/timeouts
- cookies (when needed)

Example (requests):

```python
import requests
with requests.Session() as s:
    r = s.get(url, timeout=5)
```